### Step 1 — Access and Inspect USGS 3DEP

USGS **3DEP (3D Elevation Program)** provides digital elevation models describing the ground surface. For this project, the ~10 m **1/3-arc-second DEM** is used as the terrain source.

3DEP will eventually provide static terrain context for each modeling patch, primarily through **elevation** and terrain variables derived from it, such as **slope** and **aspect**.

This step verifies that the 3DEP DEM can be accessed over Highline and inspects its band, coverage, resolution, and native projection before deriving any terrain features.

In [1]:
# Step 1A: Access and inspect USGS 3DEP over Highline

import ee

ee.Initialize(project="ee-wildfire-recovery-vasant")

# Re-load the Highline fire so this notebook is self-contained.
HIGHLINE_EVENT_ID = "ID4549011522720170729"

mtbs_boundaries = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

highline = ee.Feature(
    mtbs_boundaries
    .filter(ee.Filter.eq("Event_ID", HIGHLINE_EVENT_ID))
    .first()
)

print(
    "Fire:",
    highline.get("Incid_Name").getInfo(),
    "| Acres:",
    highline.get("BurnBndAc").getInfo()
)


# Load USGS 3DEP ~10 m DEM collection.
dep_collection = ee.ImageCollection("USGS/3DEP/10m_collection")

# Restrict the collection search to images covering Highline.
dep_highline = dep_collection.filterBounds(highline.geometry())

print("\n3DEP images intersecting Highline:",
      dep_highline.size().getInfo())

first_dep = ee.Image(dep_highline.first())

print("Bands:", first_dep.bandNames().getInfo())

print("\nProjection:")
print(first_dep.select("elevation").projection().getInfo())

Fire: HIGHLINE | Acres: 93699

3DEP images intersecting Highline: 1
Bands: ['elevation']

Projection:
{'type': 'Projection', 'crs': 'EPSG:4269', 'transform': [9.259259269220166e-05, 0, -116.0005555562933, 0, -9.259259269220166e-05, 46.00055555609498]}


#### Step 1 — Output Interpretation

One 3DEP DEM image intersects the Highline fire and contains a single `elevation` band. Slope and aspect are not original bands; they will later be derived from this elevation surface.

The DEM uses **NAD83 geographic coordinates (`EPSG:4269`)** with a pixel spacing of approximately **1/3 arc-second**, corresponding to the nominal ~10 m 3DEP product.

Importantly, **resolution and grid/projection are different concepts**:

- **Resolution** describes approximately how much ground each raster cell represents (~10 m here).
- **Projection/grid** determines the coordinate system and exact placement of those cells on Earth.

Because this DEM is stored in latitude/longitude degrees, its transform does not literally contain `10 × 10 m` cells. The source grid also does not need to match Landsat or LANDFIRE directly; all sources will later be summarized geographically onto the same common **90 × 90 m modeling grid** during integration.

### Step 2 — Inspect Highline Elevation

Before deriving terrain variables such as slope and aspect, first inspect the original 3DEP measurement: **ground elevation**.

This step checks both the numerical distribution and spatial pattern of elevation across the Highline fire to confirm that the DEM values are physically sensible and to understand how much topographic variation exists within the fire perimeter.

#### Step 2A — Elevation Distribution

Calculate elevation percentiles across the Highline fire perimeter to summarize its terrain range without relying only on the minimum and maximum.

Percentiles help distinguish the typical elevation range from small extreme areas and provide a more useful description of the overall topographic gradient.

In [2]:
# Step 2A: Inspect elevation distribution over Highline

elevation = first_dep.select("elevation")

elevation_stats = (
    elevation
    .reduceRegion(
        reducer=ee.Reducer.percentile(
            [0, 1, 10, 25, 50, 75, 90, 99, 100]
        ),
        geometry=highline.geometry(),
        scale=10,
        maxPixels=1e8
    )
    .getInfo()
)

print("Highline elevation distribution:\n")

for key, value in elevation_stats.items():
    print(f"{key}: {value:.1f} m")

Highline elevation distribution:

elevation_p0: 1483.4 m
elevation_p1: 1636.0 m
elevation_p10: 1771.8 m
elevation_p100: 2545.4 m
elevation_p25: 1883.9 m
elevation_p50: 2003.8 m
elevation_p75: 2099.9 m
elevation_p90: 2187.9 m
elevation_p99: 2340.1 m


##### Step 2A — Elevation Distribution Interpretation

Highline spans a substantial elevation gradient, from approximately **1,483 m to 2,545 m**, with a median elevation of about **2,004 m**.

Most of the fire lies within a narrower range: the 10th–90th percentiles are approximately **1,772–2,188 m**, while the 1st–99th percentiles span approximately **1,636–2,340 m**.

The percentile distribution shows that elevation varies meaningfully across Highline rather than being nearly constant, supporting its use as terrain context for modeling. Minimum and maximum values are treated cautiously because they can represent very small extreme areas.

In [3]:
# Step 2B: Visualize elevation across Highline

import geemap

Map_elevation = geemap.Map()
Map_elevation.centerObject(highline, 10)

elevation_vis = {
    "min": 1600,
    "max": 2400,
    "palette": [
        "2c7bb6",
        "abd9e9",
        "ffffbf",
        "fdae61",
        "d7191c",
    ],
}

Map_elevation.addLayer(
    elevation.clip(highline.geometry()),
    elevation_vis,
    "3DEP Elevation (m)"
)

highline_outline = ee.FeatureCollection([ee.Feature(highline)])

Map_elevation.addLayer(
    highline_outline.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Highline Perimeter"
)

Map_elevation.add_colorbar(
    elevation_vis,
    label="Elevation (m)"
)

Map_elevation

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

#### Step 2B — Elevation Map Interpretation

The elevation map shows a clear spatial topographic gradient across Highline. Lower-elevation terrain is concentrated through the central portion of the fire, while surrounding areas rise into higher-elevation terrain, with some of the highest areas toward the southern portion.

This spatial structure is consistent with the elevation percentiles from Step 2A and confirms that the 3DEP DEM is producing physically coherent terrain information across the fire perimeter.

#### Step 2 — Conclusion

Highline spans roughly **1,483–2,545 m** in elevation, with most of the fire between approximately **1,772–2,188 m** (10th–90th percentiles).

Both the distribution and map show substantial, spatially coherent elevation variation, so elevation is a meaningful candidate terrain feature for the recovery model.

### Step 3 — Derive Slope and Aspect

Elevation describes how high the terrain is, but the shape and orientation of the terrain may also influence post-fire recovery.

Using the 3DEP elevation surface, derive two additional terrain variables:

- **Slope** — how steep the terrain is.
- **Aspect** — the compass direction that the slope faces.

These are derived from differences in elevation between neighboring DEM cells rather than being separate measurements supplied directly by 3DEP.

#### Step 3A — Derive Terrain Products

Earth Engine's terrain functions use neighboring elevation cells to calculate local terrain characteristics.

#### **Slope and aspect are derived from the original, unclipped DEM so that cells near the Highline boundary can still use neighboring elevation values outside the fire perimeter.** The derived products can then be clipped to Highline for analysis and visualization.

In [4]:
# Step 3A: Derive slope and aspect from the 3DEP DEM

# Derive terrain products from the full DEM before clipping.
terrain = ee.Terrain.products(elevation)

slope = terrain.select("slope")
aspect = terrain.select("aspect")

print("Terrain product bands:")
print(terrain.bandNames().getInfo())

print("\nSlope projection:")
print(slope.projection().getInfo())

print("\nAspect projection:")
print(aspect.projection().getInfo())

Terrain product bands:
['elevation', 'slope', 'aspect', 'hillshade']

Slope projection:
{'type': 'Projection', 'crs': 'EPSG:4269', 'transform': [9.259259269220166e-05, 0, -116.0005555562933, 0, -9.259259269220166e-05, 46.00055555609498]}

Aspect projection:
{'type': 'Projection', 'crs': 'EPSG:4269', 'transform': [9.259259269220166e-05, 0, -116.0005555562933, 0, -9.259259269220166e-05, 46.00055555609498]}


##### Step 3A — Output Interpretation

The terrain operation produced four bands: the original `elevation` plus derived `slope`, `aspect`, and `hillshade`.

Slope and aspect retain the same native projection and grid as the 3DEP DEM because they are derived directly from neighboring elevation cells. `hillshade` is useful for visualization but is not currently planned as a modeling feature.

#### Step 3B — Inspect Slope

Slope measures the local steepness of the terrain and is expressed in degrees, where values near `0°` represent flat terrain and larger values represent increasingly steep slopes.

Inspect the slope distribution across Highline before mapping it to understand both typical terrain steepness and extreme areas.

In [5]:
# Step 3B.1: Inspect slope distribution over Highline

slope_stats = (
    slope
    .reduceRegion(
        reducer=ee.Reducer.percentile(
            [0, 1, 10, 25, 50, 75, 90, 99, 100]
        ),
        geometry=highline.geometry(),
        scale=10,
        maxPixels=1e8
    )
    .getInfo()
)

print("Highline slope distribution:\n")

for key, value in slope_stats.items():
    print(f"{key}: {value:.1f}°")

Highline slope distribution:

slope_p0: 0.0°
slope_p1: 1.0°
slope_p10: 4.3°
slope_p100: 64.0°
slope_p25: 8.3°
slope_p50: 13.3°
slope_p75: 18.2°
slope_p90: 23.3°
slope_p99: 33.1°


##### Step 3B.1 — Slope Distribution Interpretation

Highline contains substantial variation in terrain steepness. The median slope is approximately **13.3°**, while the central ~80% of evaluated locations fall between approximately **4.3° and 23.3°** (10th–90th percentiles).

Very steep terrain exists but is relatively uncommon: the 99th percentile is approximately **33.1°**, while the maximum of **64.0°** represents an extreme location rather than typical terrain.

This variation suggests that slope can provide meaningful terrain context beyond elevation alone.


0°   = perfectly flat  
10°  = gentle/moderate slope  
20°  = noticeably steep  
30°  = steep  
45°  = very steep  
90°  = vertical wall

In [6]:
# Step 3B.2: Visualize slope across Highline


Map_slope = geemap.Map()
Map_slope.centerObject(highline, 10)

slope_vis = {
    "min": 0,
    "max": 35,
    "palette": [
        "ffffcc",
        "c2e699",
        "78c679",
        "31a354",
        "006837",
    ],
}

Map_slope.addLayer(
    slope.clip(highline.geometry()),
    slope_vis,
    "3DEP Slope (degrees)"
)

Map_slope.addLayer(
    highline_outline.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Highline Perimeter"
)

Map_slope.add_colorbar(
    slope_vis,
    label="Slope (degrees)"
)

Map_slope

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

### Step 3B — Slope inspection

We summarized and visualized slope across the Highline fire perimeter using the 3DEP-derived terrain product.

The slope summary showed that Highline is mostly **gentle to moderately sloped terrain**, with a median slope of about **13.3°** and a 90th percentile of about **23.3°**. The map confirms this pattern visually: flatter valley-like areas appear lighter, while steeper ridges and hillsides appear darker.

This is useful because slope is a plausible predictor of wildfire recovery: steeper terrain can influence drainage, soil stability, erosion, and post-fire regeneration conditions.

### Step 3C — Aspect inspection

Next we inspect **aspect**, which describes the direction each slope faces. Aspect is relevant because north-facing and south-facing slopes can differ in solar exposure, moisture, and temperature, which may affect post-fire recovery.

For interpretation, raw aspect is useful as a map layer. For later modeling, however, aspect is better represented through derived variables such as **northness** and **eastness**, since aspect is a circular variable (0° and 360° represent the same direction).

In [7]:
# Step 3C.1 — Summarize and visualize aspect

aspect_stats = aspect.reduceRegion(
    reducer=ee.Reducer.percentile([0, 1, 10, 25, 50, 75, 90, 99, 100]),
    geometry=highline.geometry(),
    scale=10,
    maxPixels=1e8,
)

print("Highline aspect distribution:\n")
for key, value in aspect_stats.getInfo().items():
    print(f"{key}: {value:.1f}°")

aspect_vis = {
    "min": 0,
    "max": 360,
    "palette": [
        "red",      # 0   = north
        "yellow",   # 90  = east
        "green",    # 180 = south
        "cyan",     # 270 = west
        "red",      # 360 = north again
    ],
}

Map_aspect = geemap.Map()
Map_aspect.centerObject(highline, 10)

Map_aspect.addLayer(
    aspect.clip(highline.geometry()),
    aspect_vis,
    "Aspect (degrees)"
)

Map_aspect.addLayer(
    highline_outline,
    {},
    "Highline Perimeter"
)

Map_aspect.add_colorbar(
    vis_params=aspect_vis,
    label="Aspect (degrees)",
    layer_name="Aspect (degrees)"
)

Map_aspect

Highline aspect distribution:

aspect_p0: 0.3°
aspect_p1: 4.5°
aspect_p10: 44.5°
aspect_p100: 358.5°
aspect_p25: 94.5°
aspect_p50: 164.5°
aspect_p75: 268.5°
aspect_p90: 320.5°
aspect_p99: 354.5°


Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

##### Step 3C.1 — Aspect Map Interpretation

Aspect shows the compass direction faced by each slope (`0°/360° = north`, `90° = east`, `180° = south`, `270° = west`).

The Highline map shows fine-scale, terrain-following bands of different directions. This is expected because aspect can change rapidly when crossing ridges and valleys, even where elevation changes smoothly.

The spatially coherent pattern suggests that the derived aspect layer is behaving sensibly. Aspect is much more locally variable than elevation or slope and is inherently circular, so raw `0–360°` values will not be used directly as the preferred ML representation.

aspect compass convention:  
0°   = North  
90°  = East  
180° = South  
270° = West  
360° = North again  
45°  = Northeast  
135° = Southeast  
225° = Southwest  
315° = Northwest

##### Step 3C.2 — Convert Aspect to Northness and Eastness

Raw aspect is a circular variable: `0°` and `360°` both represent north, even though they appear numerically far apart.

To make aspect suitable for machine learning, convert it into two continuous directional features:

- **Northness = cos(aspect)**  
  `+1 = north-facing`, `0 = east/west-facing`, `-1 = south-facing`

- **Eastness = sin(aspect)**  
  `+1 = east-facing`, `0 = north/south-facing`, `-1 = west-facing`

This preserves the circular nature of direction and avoids treating nearby directions such as `1°` and `359°` as very different values.

In [8]:
# Step 3C.2: Convert aspect to northness and eastness

import math

# Earth Engine trigonometric functions use radians.
aspect_rad = aspect.multiply(math.pi / 180)

northness = (
    aspect_rad
    .cos()
    .rename("northness")
)

eastness = (
    aspect_rad
    .sin()
    .rename("eastness")
)

print("Northness band:", northness.bandNames().getInfo())
print("Eastness band:", eastness.bandNames().getInfo())

# Check expected numerical ranges over Highline.
direction_stats = (
    northness
    .addBands(eastness)
    .reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=highline.geometry(),
        scale=10,
        maxPixels=1e8
    )
    .getInfo()
)

print("\nDirectional feature ranges:")
for key, value in direction_stats.items():
    print(f"{key}: {value:.3f}")

Northness band: ['northness']
Eastness band: ['eastness']

Directional feature ranges:
eastness_max: 1.000
eastness_min: -1.000
northness_max: 1.000
northness_min: -1.000


##### Step 3C.2 — Output Interpretation

The derived `northness` and `eastness` features both span the expected range from approximately **-1 to +1**.

Together, these two variables preserve the full directional information contained in aspect while avoiding the circularity problem of raw `0–360°` values.

- `northness = +1` indicates north-facing terrain and `-1` indicates south-facing terrain.
- `eastness = +1` indicates east-facing terrain and `-1` indicates west-facing terrain.

Using both variables allows every compass direction to remain distinguishable in a model.

### Step 4 — Validate the Derived Terrain Features

Before defining the final 3DEP feature handoff, verify that the elevation-derived terrain variables are numerically and spatially sensible across Highline.

This step checks data coverage, expected value ranges, and important terrain edge cases—especially locations that are nearly flat, where slope direction (aspect) becomes less meaningful.

#### Step 4A — Numerical Terrain Validation

Check whether elevation, slope, and aspect have consistent coverage across Highline and confirm that their numerical ranges are physically sensible.

Because aspect represents the direction of a slope, it becomes poorly defined on nearly flat terrain. This step therefore also measures how much of Highline has very small slopes. No flat-terrain cutoff is being selected yet; the goal is only to understand whether this could become a meaningful QA issue later.

In [9]:
# Step 4A: Numerical validation of terrain layers

pixel_area_3dep = ee.Image.pixelArea()


def valid_area_m2(image):
    return (
        pixel_area_3dep
        .updateMask(image.mask())
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=highline.geometry(),
            scale=10,
            maxPixels=1e8
        )
        .get("area")
        .getInfo()
    )


# 1. Compare valid coverage

elevation_valid_m2 = valid_area_m2(elevation)
slope_valid_m2 = valid_area_m2(slope)
aspect_valid_m2 = valid_area_m2(aspect)

print("Valid terrain coverage:\n")

print(
    f"Elevation: {elevation_valid_m2 / 1e6:,.2f} km²"
)

print(
    f"Slope:     {slope_valid_m2 / 1e6:,.2f} km² "
    f"({100 * slope_valid_m2 / elevation_valid_m2:.2f}% of elevation coverage)"
)

print(
    f"Aspect:    {aspect_valid_m2 / 1e6:,.2f} km² "
    f"({100 * aspect_valid_m2 / elevation_valid_m2:.2f}% of elevation coverage)"
)


# 2. Confirm slope/aspect numerical ranges

terrain_ranges = (
    slope
    .addBands(aspect)
    .reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=highline.geometry(),
        scale=10,
        maxPixels=1e8
    )
    .getInfo()
)

print("\nTerrain ranges:")

for key, value in terrain_ranges.items():
    print(f"{key}: {value:.2f}")


# 3. Quantify nearly flat terrain

flat_lt_1_m2 = (
    pixel_area_3dep
    .updateMask(slope.lt(1))
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=highline.geometry(),
        scale=10,
        maxPixels=1e8
    )
    .get("area")
    .getInfo()
)

flat_lt_2_m2 = (
    pixel_area_3dep
    .updateMask(slope.lt(2))
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=highline.geometry(),
        scale=10,
        maxPixels=1e8
    )
    .get("area")
    .getInfo()
)

print("\nNearly flat terrain:")

print(
    f"Slope < 1°: {flat_lt_1_m2 / 1e6:,.2f} km² "
    f"({100 * flat_lt_1_m2 / slope_valid_m2:.2f}%)"
)

print(
    f"Slope < 2°: {flat_lt_2_m2 / 1e6:,.2f} km² "
    f"({100 * flat_lt_2_m2 / slope_valid_m2:.2f}%)"
)

Valid terrain coverage:

Elevation: 379.18 km²
Slope:     379.18 km² (100.00% of elevation coverage)
Aspect:    379.18 km² (100.00% of elevation coverage)

Terrain ranges:
aspect_max: 359.00
aspect_min: 0.00
slope_max: 64.00
slope_min: 0.00

Nearly flat terrain:
Slope < 1°: 1.19 km² (0.31%)
Slope < 2°: 5.88 km² (1.55%)


##### Step 4A — Numerical Validation Results

Elevation, slope, and aspect all have complete coverage across the evaluated Highline area (**379.18 km²**), indicating that the terrain derivation introduced no meaningful missing-data problem.

The derived ranges are physically sensible:

- Slope: **0–64°**
- Aspect: **0–359°**

Aspect becomes less meaningful on nearly flat terrain, but this affects only a small portion of Highline: approximately **0.31%** has slope below `1°`, and **1.55%** has slope below `2°`.

No flat-terrain cutoff is imposed during the POC. The full dataset will be examined during Phase 3 before deciding whether nearly flat areas require special handling.

#### Step 4B — Spatial Terrain Validation using Hillshade

“Do the places that our slope raster calls steep line up with terrain that visually looks steep/ridged in hillshade?”


Use hillshade derived from the same elevation surface as a visual terrain reference and compare it with the slope layer.

The goal is not to add hillshade as a model feature, but to confirm that areas mapped as steeper terrain correspond spatially to visible ridges, valleys, and terrain breaks rather than showing obvious raster artifacts.

In [10]:
# Step 4B: Compare slope with terrain hillshade

hillshade = terrain.select("hillshade")

Map_terrain_check = geemap.Map()
Map_terrain_check.centerObject(highline, 10)

# Hillshade reference
Map_terrain_check.addLayer(
    hillshade.clip(highline.geometry()),
    {
        "min": 0,
        "max": 255,
    },
    "Hillshade",
    True
)

# Slope layer for comparison
Map_terrain_check.addLayer(
    slope.clip(highline.geometry()),
    slope_vis,
    "Slope (degrees)",
    False
)

Map_terrain_check.addLayer(
    highline_outline.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Highline Perimeter"
)

Map_terrain_check

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

##### Step 4B — Spatial Validation Results

The hillshade reveals coherent ridges, valleys, and other terrain structure across Highline. When compared with the slope layer, areas of higher slope follow the same terrain breaks and steep landforms visible in the hillshade.

The agreement between the terrain visualization and the derived slope pattern provides an additional spatial sanity check that the 3DEP terrain derivatives are behaving as expected.

Hillshade is used only for visualization and validation; it is not currently planned as a modeling feature.

#### Step 5A — Proposed 3DEP Feature Contract

For the initial modeling pipeline, 3DEP will contribute four core terrain features to each future 90 × 90 m patch:

- **`elevation_mean`** — area-weighted mean ground elevation.
- **`slope_mean`** — area-weighted mean terrain slope.
- **`northness_mean`** — area-weighted mean north–south orientation derived from `cos(aspect)`.
- **`eastness_mean`** — area-weighted mean east–west orientation derived from `sin(aspect)`.

Terrain will preferably be summarized over the **eligible pre-fire forest/woodland portion** of each patch so that the terrain predictors describe the same landscape contributing to the recovery outcome.

An **area-weighted mean** gives each source value influence according to how much actual land area it contributes to the forested portion of the 90 m patch. If all source pixels were equal-area, perfectly aligned, and fully included, an ordinary pixel mean would already provide equivalent weighting because values occurring in more pixels would naturally count more often.

Explicit area weighting is more general because the 90 m grid, the native 3DEP grid/projection, and forest-mask boundaries do not necessarily align. Individual 3DEP cells may therefore contribute only part of their area to a patch.

Together, the four features represent elevation, steepness, and directional exposure without using raw circular aspect degrees. Additional terrain-variability features such as elevation or slope standard deviation can be evaluated later during Phase 3/EDA if they add useful information.

#### Step 5B — 3DEP Integration Handoff and Open Decisions

When the common **90 × 90 m modeling grid** is created during Phase 2 integration, the 3DEP terrain layers will be summarized over the eligible pre-fire forest/woodland portion of each patch.

Planned core features:

- `elevation_mean`
- `slope_mean`
- `northness_mean`
- `eastness_mean`

These will be calculated using geographic area-weighted summaries so that partially overlapping source cells contribute in proportion to the land area they represent inside each forested patch.

Several choices remain open for Phase 3 and EDA:

- Whether additional terrain-heterogeneity features such as elevation or slope variability improve prediction.
- Whether nearly flat terrain requires special handling for northness/eastness.
- Final implementation details for aggregation onto the common 90 m grid.
- Validation of the feature distributions across the full Western U.S. fire population.

The current POC establishes the terrain feature logic without prematurely optimizing these choices to Highline alone.

In [11]:
# Step 6A: Save curated 3DEP POC figures

from pathlib import Path
import urllib.request


# Find repository root.
cwd = Path.cwd()
repo_root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

figure_dir = repo_root / "outputs" / "figures" / "3dep"
figure_dir.mkdir(parents=True, exist_ok=True)


# Highline perimeter overlay.
outline = (
    ee.Image()
    .byte()
    .paint(highline_outline, 1, 2)
    .selfMask()
    .visualize(palette=["000000"])
)


def save_ee_png(image, vis_params, filename):
    """Render an Earth Engine image and save a clean PNG."""

    rendered = (
        image
        .clip(highline.geometry())
        .visualize(**vis_params)
        .blend(outline)
    )

    url = rendered.getThumbURL({
        "region": highline.geometry(),
        "dimensions": 1600,
        "format": "png",
    })

    output_path = figure_dir / filename
    urllib.request.urlretrieve(url, output_path)

    print(f"Saved: {output_path.relative_to(repo_root)}")


# 1. Elevation
save_ee_png(
    elevation,
    elevation_vis,
    "highline_elevation.png",
)


# 2. Slope
save_ee_png(
    slope,
    slope_vis,
    "highline_slope.png",
)


# 3. Aspect
save_ee_png(
    aspect,
    aspect_vis,
    "highline_aspect.png",
)


# 4. Hillshade — validation / terrain reference only
save_ee_png(
    hillshade,
    {
        "min": 0,
        "max": 255,
    },
    "highline_hillshade.png",
)

Saved: outputs\figures\3dep\highline_elevation.png
Saved: outputs\figures\3dep\highline_slope.png
Saved: outputs\figures\3dep\highline_aspect.png
Saved: outputs\figures\3dep\highline_hillshade.png
